# CSE 251B FineWeb-Edu Pipeline

This notebook prepares FineWeb-Edu shards, trains the nano model, runs the TA evaluation script, and packages the submission files under `results/<timestamp>` before copying them to Drive.

In [1]:
!nvidia-smi

## 1. Mount Drive And Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import time

DRIVE_ROOT = Path('/content/drive/MyDrive/251B')
DRIVE_DATASET_DIR = DRIVE_ROOT / 'dataset' / 'fineweb_edu'
DRIVE_RESULTS_DIR = DRIVE_ROOT / 'results'

REPO_DIR = Path('/content/milestone')
LOCAL_DATASET_DIR = REPO_DIR / 'data' / 'fineweb_edu'

RUN_NAME = 'fineweb_edu_nano_' + time.strftime('%Y%m%d_%H%M%S')
LOCAL_RESULTS_DIR = REPO_DIR / 'results' / RUN_NAME
DRIVE_RUN_DIR = DRIVE_RESULTS_DIR / RUN_NAME

DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Drive dataset:', DRIVE_DATASET_DIR)
print('Drive results:', DRIVE_RESULTS_DIR)
print('Repo:', REPO_DIR)
print('Local dataset:', LOCAL_DATASET_DIR)
print('Local run results:', LOCAL_RESULTS_DIR)
print('Drive run results:', DRIVE_RUN_DIR)

## 2. Prepare Repository

In [3]:
!rm -rf /content/milestone
!git clone -b v1 https://github.com/FufenNan/milestone.git /content/milestone
%cd /content/milestone
!git status --short --branch

LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Install Dependencies

In [4]:
!pip install -q datasets tiktoken tqdm

## 4. Copy Or Prepare FineWeb-Edu

In [5]:
DATASET_LOG = LOCAL_RESULTS_DIR / 'dataset_prepare.log'

def log(msg):
    line = f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {msg}'
    print(line)
    with open(DATASET_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def count_shards(path):
    path = Path(path)
    train = sorted(path.glob('fineweb_train_*.bin'))
    val = sorted(path.glob('fineweb_val_*.bin'))
    return len(train), len(val)

def shard_bytes(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.glob('fineweb_*.bin'))

def has_ready_dataset(path, min_train_shards=1):
    train_count, val_count = count_shards(path)
    return train_count >= min_train_shards and val_count >= 1

def sync_dir(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(['rsync', '-ah', '--info=progress2', f'{src}/', f'{dst}/'], check=True)
    except FileNotFoundError:
        shutil.copytree(src, dst, dirs_exist_ok=True)

SHARD_SIZE = 100_000_000
MAX_SHARDS = 22
MIN_TRAIN_SHARDS = MAX_SHARDS - 1
NUM_PROC = max(1, (os.cpu_count() or 2) // 2)

drive_train, drive_val = count_shards(DRIVE_DATASET_DIR)
log(f'Drive shards: train={drive_train}, val={drive_val}, size={shard_bytes(DRIVE_DATASET_DIR) / 1e9:.2f} GB')

if has_ready_dataset(DRIVE_DATASET_DIR, MIN_TRAIN_SHARDS):
    log('Copying FineWeb-Edu from Drive to repo data/fineweb_edu...')
    sync_dir(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
else:
    log('Drive dataset is incomplete. Preparing FineWeb-Edu under repo data/fineweb_edu...')
    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        'python', '-u', 'data/fineweb.py',
        '--data-root', str(LOCAL_DATASET_DIR),
        '--streaming',
        '--shard-size', str(SHARD_SIZE),
        '--max-shards', str(MAX_SHARDS),
        '--num-proc', str(NUM_PROC),
    ]
    log('Running: ' + ' '.join(cmd))
    with open(DATASET_LOG, 'a', encoding='utf-8') as log_file:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'Dataset preparation failed with exit code {ret}. See {DATASET_LOG}')
    log('Copying prepared FineWeb-Edu shards back to Drive...')
    sync_dir(LOCAL_DATASET_DIR, DRIVE_DATASET_DIR)

local_train, local_val = count_shards(LOCAL_DATASET_DIR)
log(f'Local shards: train={local_train}, val={local_val}, size={shard_bytes(LOCAL_DATASET_DIR) / 1e9:.2f} GB')
assert has_ready_dataset(LOCAL_DATASET_DIR, 1), 'FineWeb-Edu dataset is not ready in repo data/fineweb_edu.'

## 5. Train

In [6]:
TRAIN_STDOUT = LOCAL_RESULTS_DIR / 'train_stdout.log'

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

cmd = ['python', '-u', 'train.py', '--config', 'config/config.py']
# cmd = ['python', '-u', 'train.py', '--config', 'config/config_10000.py']
print('Run name:', RUN_NAME)
print('Command:', ' '.join(cmd))

with open(TRAIN_STDOUT, 'w', encoding='utf-8') as log_file:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}. See {TRAIN_STDOUT}')

## 6. Run TA Evaluation

In [7]:
LOCAL_CHECKPOINT = REPO_DIR / 'checkpoints' / 'checkpoint.pt'
EVAL_DATA = REPO_DIR / 'val.bin'
TA_EVAL_JSON = LOCAL_RESULTS_DIR / 'eval_val.json'

assert LOCAL_CHECKPOINT.exists(), f'Missing checkpoint: {LOCAL_CHECKPOINT}'
assert EVAL_DATA.exists(), f'Missing eval data: {EVAL_DATA}'

cmd = [
    'python', '-u', 'evaluate.py',
    '--model_dir', str(REPO_DIR),
    '--checkpoint_filename', 'checkpoints/checkpoint.pt',
    '--data', str(EVAL_DATA),
    '--device', 'cuda',
    '--batch_size', '1',
    '--output_json', str(TA_EVAL_JSON),
]
print('Command:', ' '.join(cmd))

proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'TA evaluation failed with exit code {ret}.')

## 7. Package Results And Copy To Drive

In [8]:
submission_files = {
    REPO_DIR / 'checkpoints' / 'checkpoint.pt': LOCAL_RESULTS_DIR / 'checkpoint.pt',
    REPO_DIR / 'config' / 'config.py': LOCAL_RESULTS_DIR / 'config.py',
    # REPO_DIR / 'config' / 'config_10000.py': LOCAL_RESULTS_DIR / 'config.py',
    REPO_DIR / 'model.py': LOCAL_RESULTS_DIR / 'model.py',
}

optional_logs = [
    REPO_DIR / 'checkpoints' / 'train.log',
    REPO_DIR / 'results' / 'train.log',
    REPO_DIR / 'checkpoints' / 'checkpoint_metadata.pt',
]

for src, dst in submission_files.items():
    assert src.exists(), f'Missing expected file: {src}'
    shutil.copy2(src, dst)

for src in optional_logs:
    if src.exists():
        shutil.copy2(src, LOCAL_RESULTS_DIR / src.name)

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
sync_dir(LOCAL_RESULTS_DIR, DRIVE_RUN_DIR)

print('Local results:', LOCAL_RESULTS_DIR)
print('Drive results:', DRIVE_RUN_DIR)
print('Packaged files:')
for path in sorted(LOCAL_RESULTS_DIR.iterdir()):
    print(' -', path.name)

# 8. Exit Colab

In [9]:
from google.colab import runtime
runtime.unassign()